# ForensiCore Audio Training (Kaggle)
This notebook preprocesses FakeAVCeleb audio and trains the audio model.
Clone branch: ide-fakeav

In [ ]:
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('tensorflow-addons is optional for this project.')

In [ ]:
# Clone this branch from GitHub: ide-fakeav
!rm -rf /kaggle/working/swin-model
!git clone -b ide-fakeav https://github.com/RafiaMushtaq04/swin-model.git /kaggle/working/swin-model

import os
import random
import subprocess
import sys

from pathlib import Path
import importlib.util
import numpy as np
import tensorflow as tf

requirements_file = Path('/kaggle/working/swin-model/requirements.txt')
if requirements_file.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_file)], check=True)

seed = 3
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

try:
    from tensorflow.keras import mixed_precision
    if tf.config.list_physical_devices('GPU'):
        mixed_precision.set_global_policy('mixed_float16')
        print('Mixed precision enabled')
except Exception as exc:
    print('Mixed precision not enabled:', exc)

# FakeAVCeleb audio preprocessing (log-mel PNGs)
subprocess.run([
    sys.executable,
    '/kaggle/working/swin-model/ForensiCore-fakeav-audio-preprocess.py',
    '--root', '/kaggle/input/datasets/uzairfar00q/fakeavceleb/FakeAVCeleb',
    '--out', '/kaggle/working/fakeav_audio_png',
    '--train-split', '0.8',
    '--val-split', '0.1',
    '--test-split', '0.1',
], check=True)

launcher_path = Path('/kaggle/working/swin-model/ForensiCore-train.py')
if not launcher_path.exists():
    raise FileNotFoundError(f'Missing launcher script: {launcher_path}')

spec = importlib.util.spec_from_file_location('forensicore_launcher', launcher_path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

module.run_audio_pipeline()